# Face Model Lab: Video Bearbeitung & Face Blurring

Dieses Notebook nutzt die eigenständige Datei `step08_blur_video.py`. Es ist für Smoke-Tests, Modellvergleich im Video und finale Blur-Läufe gedacht.

In [ ]:
from pathlib import Path
import os
import re
import subprocess
import sys

from IPython.display import Video, display

def find_project_paths():
    cwd = Path.cwd().resolve()
    if (cwd / "face_model_lab" / "step08_blur_video.py").exists():
        return cwd, cwd / "face_model_lab"
    if (cwd / "step08_blur_video.py").exists():
        return cwd.parent, cwd
    raise RuntimeError("Notebook bitte aus dem Projekt-Root oder aus face_model_lab starten.")

ROOT, LAB = find_project_paths()
os.environ.setdefault("MPLCONFIGDIR", str(ROOT / "matplotlib_cache"))
PYTHON = sys.executable
VIDEOS = ROOT / "Videos"
MODEL_DIRS = [ROOT / "trained_models", LAB / "trained_models"]

print("Root:", ROOT)
print("Lab:", LAB)
print("Videos:", VIDEOS)
print("Model dirs:", [p for p in MODEL_DIRS if p.exists()])
print("Python:", PYTHON)


In [ ]:
def run_cmd(args):
    print("$", " ".join(map(str, args)))
    result = subprocess.run(args, cwd=ROOT, text=True)
    if result.returncode != 0:
        raise RuntimeError(f"Command failed with exit code {result.returncode}")


## Modell und Video auswählen

Du kannst `.pt`-Modelle von Ultralytics oder `.pth`-Modelle von Torchvision verwenden. Für eigene trainierte Modelle wähle ein Modell aus `trained_models/`; der Output-Name enthält automatisch den Modellnamen.

In [ ]:
def slugify(value):
    return re.sub(r"[^A-Za-z0-9_.-]+", "_", value).strip("_")

trained_model_candidates = [
    *[p for model_dir in MODEL_DIRS for p in sorted(model_dir.glob("yolo*_bs*_red*_ep*.pt"))],
    *[p for model_dir in MODEL_DIRS for p in sorted(model_dir.glob("rtdetr*_bs*_red*_ep*.pt"))],
    *[p for model_dir in MODEL_DIRS for p in sorted(model_dir.glob("fasterrcnn*_bs*_red*_ep*.pth"))],
    *[p for model_dir in MODEL_DIRS for p in sorted(model_dir.glob("retinanet*_bs*_red*_ep*.pth"))],
    *[p for model_dir in MODEL_DIRS for p in sorted(model_dir.glob("fcos*_bs*_red*_ep*.pth"))],
    *[p for model_dir in MODEL_DIRS for p in sorted(model_dir.glob("yolo*_bs*_ep*.pt"))],
    *[p for model_dir in MODEL_DIRS for p in sorted(model_dir.glob("rtdetr*_bs*_ep*.pt"))],
    *[p for model_dir in MODEL_DIRS for p in sorted(model_dir.glob("fasterrcnn*_bs*_ep*.pth"))],
    *[p for model_dir in MODEL_DIRS for p in sorted(model_dir.glob("retinanet*_bs*_ep*.pth"))],
    *[p for model_dir in MODEL_DIRS for p in sorted(model_dir.glob("fcos*_bs*_ep*.pth"))],
]
model_candidates = [
    ROOT / "face_yolov8m.pt",
    LAB / "face_yolov8m.pt",
    *trained_model_candidates,
    ROOT / "yolov8m.pt",
    LAB / "yolov8m.pt",
]
seen = set()
model_candidates = [p for p in model_candidates if not (p in seen or seen.add(p))]
model_path = next((p for p in model_candidates if p.exists()), None)

# Hier nur den Dateinamen anpassen, wenn du ein anderes Video aus Videos/ verwenden willst.
input_video = VIDEOS / "Feuerwehr_progressiv.mp4"

# Eindeutiger Output pro Video und Modell, z.B. Feuerwehr_progressiv__blur__yolo_yolov8m_widerface_rocm_bs2_red1_ep15_smoke.mp4
output_dir = VIDEOS / "lab_outputs"
model_tag = slugify(model_path.stem) if model_path else "kein_modell"
output_video = output_dir / f"{slugify(input_video.stem)}__blur__{model_tag}_smoke.mp4"

CONFIDENCE_THRESHOLD = 0.25
IMAGE_SIZE = 640
FRAME_SKIP = 2
MAX_FRAMES = 300
DEINTERLACE_MODE = "smoke"  # "smoke", "auto", "always", "never"
BACKGROUND_THRESHOLD_PERCENT = 0.0
PADDING_FACTOR = 0.5
BLUR_KERNEL = 99
MASK_KERNEL = 71

print("Model:", model_path if model_path else "kein .pt- oder .pth-Modell gefunden")
print("Input:", input_video)
print("Output:", output_video)


## Smoke-Test: wenige Frames blurren

In [ ]:
if model_path is None:
    print("Überspringe Video-Test: Es wurde kein .pt- oder .pth-Modell gefunden.")
else:
    run_cmd([
        PYTHON, "face_model_lab/step08_blur_video.py",
        "--model", str(model_path),
        "--input", str(input_video),
        "--output", str(output_video),
        "--conf", str(CONFIDENCE_THRESHOLD),
        "--imgsz", str(IMAGE_SIZE),
        "--frame-skip", str(FRAME_SKIP),
        "--max-frames", str(MAX_FRAMES),
        "--deinterlace", DEINTERLACE_MODE,
        "--background-threshold-percent", str(BACKGROUND_THRESHOLD_PERCENT),
        "--padding-factor", str(PADDING_FACTOR),
        "--blur-kernel", str(BLUR_KERNEL),
        "--mask-kernel", str(MASK_KERNEL),
    ])


## Ausgabe anzeigen

In [ ]:
if output_video.exists():
    display(Video(str(output_video), embed=False, width=720))
else:
    print("Output wurde nicht gefunden:", output_video)


## Finaler Lauf

Für ein komplettes Video `--max-frames` entfernen. Höheres `imgsz` findet kleine Gesichter besser, kostet aber mehr VRAM/Zeit.

In [ ]:
if model_path is None:
    print("Überspringe Video-Test: Es wurde kein .pt- oder .pth-Modell gefunden.")
else:
    run_cmd([
        PYTHON, "face_model_lab/step08_blur_video.py",
        "--model", str(model_path),
        "--input", str(input_video),
        "--output", str(output_video),
        "--conf", str(CONFIDENCE_THRESHOLD),
        "--imgsz", str(IMAGE_SIZE),
        "--frame-skip", str(FRAME_SKIP),
        "--max-frames", str(MAX_FRAMES),
        "--deinterlace", DEINTERLACE_MODE,
        "--background-threshold-percent", str(BACKGROUND_THRESHOLD_PERCENT),
        "--padding-factor", str(PADDING_FACTOR),
        "--blur-kernel", str(BLUR_KERNEL),
        "--mask-kernel", str(MASK_KERNEL),
    ])
